# TEM Diffraction Pattern Indexing: The Round Trip

Indexing a selected-area electron diffraction pattern is the routine question asked in front
of a microscope: *which phase is this, which zone axis am I looking down, and what are the
indices of those spots?* PyTex answers it with `solve_saed_pattern`, using classical
ratio/angle indexing on the spot geometry alone.

This tutorial does not simply demonstrate that call. It **closes the loop**: simulate a
pattern from a known phase and a known zone axis, hand the resulting spot positions to the
solver as if they had been picked off a micrograph, and check that what comes back is what
went in. Two canonical cases run throughout — **nickel**, face-centred cubic, and
**zirconium**, hexagonal close-packed — because the two lattices fail in different ways and a
method that works on cubic alone has not been tested.

### What a round trip does and does not prove

A round trip tests the *pair* of models for mutual consistency. If the forward simulation and
the inverse indexing agree, then given that the forward model is independently validated —
against pymatgen powder baselines and tabulated $d$-spacings, in
`tests/unit/test_diffraction_external_baselines.py` — the inverse is validated too.

What it cannot detect is an error shared by both directions. A sign convention that is wrong
in the same way in the simulator and the solver would round-trip perfectly. So the round trip
is a strong *internal-consistency* test and not a substitute for the external baselines; both
are needed, and this notebook says which is which rather than overselling one of them.

### What you will see

1. The forward model, and the geometry it encodes.
2. Turning a simulation into a "measurement" — and what a measurement is allowed to know.
3. The indexing algorithm, stated as an algorithm.
4. Round trips for four Ni zones and four Zr zones.
5. Why the recovered zone axis is often *not* the one you typed, and why that is correct.
6. Recovering the orientation, not just the zone.
7. What noise does, and where the method breaks.
8. Phase discrimination, including a case where one pattern is not enough.

## 0. Setup: nickel and zirconium

Both phases come from the pinned CIF fixtures, so the lattice parameters are checksummed
rather than typed in.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

from pytex import (
    Orientation,
    ZoneAxis,
    crystal_frame,
    format_direction_indices,
    format_plane_indices,
    get_phase_fixture,
    specimen_frame,
)
from pytex.core.miller import MillerDirection
from pytex.diffraction.kinematic import KinematicSimulationConfig, simulate_zone_axis_spots
from pytex.diffraction.solving import (
    MeasuredSAEDPattern,
    MeasuredSpot,
    PatternCalibration,
    solve_saed_pattern,
)

CRYSTAL = crystal_frame()
SPECIMEN = specimen_frame()

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="No _symmetry_equiv_pos_as_xyz.*")
    warnings.filterwarnings("ignore", message="Issues encountered while parsing CIF.*")
    NICKEL = get_phase_fixture("ni_fcc").load_phase(crystal_frame=CRYSTAL)
    ZIRCONIUM = get_phase_fixture("zr_hcp").load_phase(crystal_frame=CRYSTAL)

for phase in (NICKEL, ZIRCONIUM):
    lattice = phase.lattice
    print(f"{phase.name:<15} a = {lattice.a:.4f} A  c = {lattice.c:.4f} A  "
          f"point group {phase.symmetry.point_group}  "
          f"space group {phase.space_group_symbol}")
print(f"\nZr c/a = {ZIRCONIUM.lattice.c / ZIRCONIUM.lattice.a:.4f} "
      f"(ideal close packing is {np.sqrt(8.0 / 3.0):.4f})")

## 1. The forward model

`simulate_zone_axis_spots` places a reflection $\mathbf{g}$ on the detector at

$$\mathbf{r}_{\text{mm}} = (L\lambda)\,\mathbf{g}_\perp,$$

where $\mathbf{g}_\perp$ is the component of $\mathbf{g}$ perpendicular to the zone axis and
$L\lambda$ is the camera constant — the one quantity a real microscope is calibrated for.
Which reflections appear is decided by the **excitation error**

$$s_g = g_z - \tfrac{1}{2}\lambda\,|\mathbf{g}|^{2},$$

not by the integer zone law. The distinction matters: the $-\lambda g^2/2$ term is the
curvature of the Ewald sphere, and selecting on $|s_g|$ rather than on $\mathbf{g}\cdot
[uvw] = 0$ is what lets an irrational zone axis be handled at all.

Systematic absences come from the phase's space group. For FCC nickel that removes every
reflection with mixed-parity indices; for HCP zirconium it removes the $\{000\ell\}$ with
odd $\ell$ and thins the rest.

In [ ]:
CONFIG = KinematicSimulationConfig(
    beam_energy_kev=200.0,
    camera_constant_mm_angstrom=180.0,
    max_index=4,
    g_max_inv_angstrom=1.2,
)


def simulate(phase, uvw):
    return simulate_zone_axis_spots(phase, ZoneAxis(np.asarray(uvw), phase=phase), config=CONFIG)


def plot_pattern(ax, table, title):
    coordinates = np.asarray(table.detector_mm)
    intensity = np.asarray(table.intensity)
    ax.scatter([0.0], [0.0], s=90, facecolors="none", edgecolors="black", lw=1.2)
    ax.scatter(coordinates[:, 0], coordinates[:, 1], s=25 + 120 * intensity,
               c=intensity, cmap="magma", vmin=0.0, vmax=1.0)
    for (x, y), hkl in zip(coordinates, table.hkl):
        if np.hypot(x, y) < 0.62 * np.abs(coordinates).max():
            ax.annotate(format_plane_indices(tuple(int(v) for v in hkl), style="mathtext"),
                        (x, y), textcoords="offset points", xytext=(5, 4), fontsize=7)
    extent = 1.15 * np.abs(coordinates).max()
    ax.set_xlim(-extent, extent); ax.set_ylim(-extent, extent)
    ax.set_aspect("equal"); ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


fig, axes = plt.subplots(1, 2, figsize=(10.5, 5.2))
plot_pattern(axes[0], simulate(NICKEL, (0, 0, 1)), "Ni (fcc), zone [001]")
plot_pattern(axes[1], simulate(ZIRCONIUM, (0, 0, 1)), "Zr (hcp), zone [0001]")
fig.suptitle("simulated zone-axis patterns; open circle = transmitted beam", fontsize=11)
plt.show()

The two patterns already show the difference the two lattices make. The FCC $[001]$ pattern is
a square net of $\{200\}$ and $\{220\}$: no $\{100\}$, because mixed parity is forbidden. The
HCP $[0001]$ pattern is a hexagonal net — and the innermost hexagon is $\{11\bar{2}0\}$, not
$\{10\bar{1}0\}$, because the two-atom basis of the HCP structure extinguishes the latter.
Reading the *absences* is often how a phase is identified.

## 2. From a simulation to a "measurement"

A solver must not be handed anything an experimenter would not have. What comes off a
micrograph is a list of spot positions in some units, plus enough calibration to convert them
into reciprocal-space lengths. `MeasuredSAEDPattern` is exactly that and nothing more.

Two constraints in the type are worth pausing on, because both encode a real mistake:

- **The transmitted beam is not a spot.** It is the calibration's `centre`, and every
  position is taken relative to it. A "spot" at the centre would have no direction, and the
  type rejects it.
- **Intensities are optional and are never used.** Ratio/angle indexing decides on geometry
  alone. Kinematic intensities are unreliable — dynamical scattering and double diffraction
  see to that — so letting them influence the answer would import an error the geometry does
  not have.

In [ ]:
CALIBRATION = PatternCalibration(units="mm", camera_constant_mm_angstrom=180.0)


def as_measurement(table, name, *, rng=None, noise_mm=0.0):
    # Spot positions only, as if picked off a micrograph.

    coordinates = np.asarray(table.detector_mm, dtype=np.float64)
    if noise_mm > 0.0:
        coordinates = coordinates + rng.normal(scale=noise_mm, size=coordinates.shape)
    return MeasuredSAEDPattern(
        name=name,
        spots=tuple(MeasuredSpot(position=(float(x), float(y))) for x, y in coordinates),
        calibration=CALIBRATION,
    )


measurement = as_measurement(simulate(NICKEL, (0, 0, 1)), "ni_001")
print(f"{len(measurement)} spots, no indices, no phase, no zone axis attached.")
print("d-spacings the solver will work from (A):")
print(np.round(np.sort(measurement.d_spacings_angstrom())[::-1][:6], 4))
print("\nNi lattice parameter is", f"{NICKEL.lattice.a:.4f} A, so d_200 =",
      f"{NICKEL.lattice.a / 2.0:.4f} A and d_220 =", f"{NICKEL.lattice.a / np.sqrt(8.0):.4f} A")

## 3. The indexing algorithm

Ratio/angle indexing is old, simple, and robust, and it uses nothing but the geometry.

> **Algorithm — ratio/angle indexing of a zone-axis pattern**
>
> **Input:** measured in-plane vectors $\mathbf{g}^{\text{obs}}_i$; candidate phases;
> an index bound; a relative length tolerance $\varepsilon_L$ and an angular tolerance
> $\varepsilon_\theta$.
>
> 1. **Seed.** Take the two shortest non-collinear measured vectors,
>    $(\mathbf{g}^{\text{obs}}_a, \mathbf{g}^{\text{obs}}_b)$.
> 2. **Enumerate.** For each candidate phase, list all allowed reflections within the index
>    bound, applying the space-group absences.
> 3. **Match.** A calculated pair $(\mathbf{g}_1, \mathbf{g}_2)$ is admissible when
>    $\bigl||\mathbf{g}_k| - |\mathbf{g}^{\text{obs}}|\bigr| \le \varepsilon_L
>    |\mathbf{g}^{\text{obs}}|$ for both, **and** the interplanar angle matches within
>    $\varepsilon_\theta$. Lengths alone are not enough: many reflections share a $d$-spacing.
> 4. **Zone axis.** $[uvw] \parallel \mathbf{g}_1 \times \mathbf{g}_2$, reduced to integers.
> 5. **Orientation.** Build right-handed triads from the calculated and the observed pairs and
>    align them; the result is the crystal-to-pattern rotation.
> 6. **Index the rest.** Project every remaining calculated reflection into the pattern plane
>    and assign it to the nearest measured spot within tolerance. Record the residual.
> 7. **Rank.** Order candidate solutions by matched fraction first, then by mean residual.
>
> **Output:** ranked solutions, each with phase, zone axis, orientation, per-spot indices and
> residuals — and a verdict on whether the best is unambiguous.

Step 7 puts matched fraction ahead of residual deliberately. A solution that explains *every*
spot with moderate residuals is a better answer than one that explains half of them perfectly,
because the latter is usually a coincidence on a sub-lattice.

## 4. Round trip: nickel

Four zones, each simulated and then solved with no knowledge of how it was made.

In [ ]:
def round_trip(phase, uvw, candidates, *, rng=None, noise_mm=0.0, **solver):
    table = simulate(phase, uvw)
    measurement = as_measurement(table, f"{phase.name}_{uvw}", rng=rng, noise_mm=noise_mm)
    report = solve_saed_pattern(measurement, candidates, max_index=4, **solver)
    return table, measurement, report


def zone_equivalent(phase, first, second):
    # Whether two zone axes are the same axis under the crystal symmetry, up to sign.

    orbit, _ = MillerDirection.from_uvw(np.asarray(first), phase=phase).symmetry_equivalent_indices()
    family = np.asarray(orbit).reshape(-1, 3)
    target = np.asarray(second, dtype=np.int64)
    return bool(np.any(np.all(family == target, axis=1) | np.all(family == -target, axis=1)))


NICKEL_ZONES = [(0, 0, 1), (0, 1, 1), (1, 1, 1), (1, 1, 2)]

print(f"{'typed':<10} {'spots':>5}  {'recovered':<22} {'match':>6} {'residual':>11}  "
      f"{'equiv?':>6} {'unique?':>7}")
for uvw in NICKEL_ZONES:
    table, measurement, report = round_trip(NICKEL, uvw, [NICKEL])
    best = report.best()
    recovered = np.asarray(best.zone_axis.indices, dtype=np.int64)
    print(f"{str(uvw):<10} {len(measurement):>5}  "
          f"{best.phase_name + ' ' + best.zone_axis_label:<22} "
          f"{100 * best.matched_fraction:5.0f}% {best.mean_residual_inv_angstrom:11.2e}  "
          f"{str(zone_equivalent(NICKEL, uvw, recovered)):>6} {str(report.is_conclusive):>7}")

Every spot indexed, residuals at the level of floating-point round-off, and every recovered
zone axis in the same symmetry family as the one that was typed. That last column is the point
of the next section.

## 5. Round trip: zirconium, and the four-index question

HCP zirconium exercises everything cubic nickel does not: a non-cubic metric, a two-atom
basis with its own extinctions, and the four-index Miller-Bravais notation the literature
uses for hexagonal directions. PyTex stores three indices internally and converts for
display — `[uvw]` to `[uvtw]` with $t = -(u+v)$ after the $\tfrac{1}{3}$ scaling — so the
same solver runs unchanged.

In [ ]:
ZIRCONIUM_ZONES = [(0, 0, 1), (1, 1, 0), (1, 0, 0), (2, 1, 0)]

print(f"{'typed [uvw]':<12} {'typed [uvtw]':<16} {'spots':>5}  {'recovered [uvtw]':<18} "
      f"{'match':>6} {'residual':>11} {'equiv?':>7}")
for uvw in ZIRCONIUM_ZONES:
    table, measurement, report = round_trip(ZIRCONIUM, uvw, [ZIRCONIUM])
    best = report.best()
    recovered = np.asarray(best.zone_axis.indices, dtype=np.int64)
    typed_four = MillerDirection.from_uvw(np.asarray(uvw), phase=ZIRCONIUM).to_miller_bravais()
    got_four = MillerDirection.from_uvw(recovered, phase=ZIRCONIUM).to_miller_bravais()
    print(f"{str(uvw):<12} "
          f"{format_direction_indices(tuple(int(v) for v in typed_four.reduced_indices), style='plain'):<16} "
          f"{len(measurement):>5}  "
          f"{format_direction_indices(tuple(int(v) for v in got_four.reduced_indices), style='plain'):<18} "
          f"{100 * best.matched_fraction:5.0f}% {best.mean_residual_inv_angstrom:11.2e} "
          f"{str(zone_equivalent(ZIRCONIUM, uvw, recovered)):>7}")

## 6. Why the recovered zone axis is often not the one you typed

Look at the zirconium table: $[10\bar{1}0]$ went in and something else came back, yet the
"equiv?" column says the answer is right. Both statements are true, and the reason is the
whole reason a round trip has to be checked *up to symmetry* rather than by string comparison.

A diffraction pattern is invariant under two things the solver cannot see through:

1. **Crystal symmetry.** Every zone axis in the same symmetry orbit produces an *identical*
   pattern. There is no measurement that distinguishes them, so reporting any member is
   correct; reporting one particular member would be a convention, not information.
2. **Friedel's law.** For a centrosymmetric reflection set, $\mathbf{g}$ and $-\mathbf{g}$ have
   the same intensity, so a zone axis and its reverse give the same pattern. A single SAED
   pattern therefore cannot tell $[uvw]$ from $[\bar{u}\bar{v}\bar{w}]$.

`PatternSolutionReport.describe()` says the second one out loud rather than presenting one
sense as *the* answer.

In [ ]:
target = (1, 0, 0)
_, _, report = round_trip(ZIRCONIUM, target, [ZIRCONIUM])
recovered = np.asarray(report.best().zone_axis.indices, dtype=np.int64)

orbit, _ = MillerDirection.from_uvw(np.asarray(target), phase=ZIRCONIUM).symmetry_equivalent_indices()
family = np.unique(np.asarray(orbit).reshape(-1, 3), axis=0)
print(f"typed {target}, recovered {tuple(int(v) for v in recovered)}")
print(f"\nthe symmetry orbit of {target} under {ZIRCONIUM.symmetry.point_group} "
      f"has {len(family)} members:")
print(family)
print("\nEvery one of them produces the identical pattern, so all are correct answers.")

In [ ]:
# And the two senses of one zone axis: simulate both, compare the spot sets.
forward = simulate(ZIRCONIUM, (1, 1, 0))
reverse = simulate(ZIRCONIUM, (-1, -1, 0))
forward_radii = np.sort(np.linalg.norm(np.asarray(forward.detector_mm), axis=1))
reverse_radii = np.sort(np.linalg.norm(np.asarray(reverse.detector_mm), axis=1))

print(f"spots down [110]  : {len(forward.hkl)}")
print(f"spots down [-1-10]: {len(reverse.hkl)}")
print("largest difference between the sorted spot radii:",
      f"{float(np.abs(forward_radii - reverse_radii).max()):.2e} mm")
print("\nThe two patterns are the same measurement. Distinguishing the sense needs")
print("something a single pattern does not carry -- a second zone, or dynamical")
print("intensities that break Friedel's law.")

## 7. The orientation, not just the zone axis

`PatternSolution.orientation` is the crystal-to-pattern rotation: it carries crystal Cartesian
vectors into the stored pattern's own axes. That is strictly more than the zone axis — it also
fixes the rotation *about* the beam, which is what the in-plane indexing determines.

The simulation knows the true answer: its `basis` has the detector $u$, $v$ axes and the zone
axis as columns, so the true crystal-to-pattern rotation is $\mathsf{basis}^{\mathsf{T}}$.
Comparing the two must be done **symmetry-aware**, for exactly the reason of section 6: the
recovered rotation may differ from the true one by a crystal symmetry operation and still be
the same physical answer.

In [ ]:
def as_orientation(matrix, phase):
    return Orientation.from_matrix(
        np.asarray(matrix), specimen_frame=SPECIMEN, phase=phase, crystal_frame=CRYSTAL
    )


print(f"{'phase':<15} {'zone':<10} {'raw angle':>10} {'best solution':>14} "
      f"{'best of all':>12} {'ranked':>7}")
for phase, zones in ((NICKEL, NICKEL_ZONES), (ZIRCONIUM, ZIRCONIUM_ZONES)):
    for uvw in zones:
        table, _, report = round_trip(phase, uvw, [phase])
        truth = as_orientation(np.asarray(table.basis).T, phase)

        deviations = [
            truth.misorientation_to(
                as_orientation(solution.orientation.as_matrix(), phase)
            ).disorientation().angle_deg
            for solution in report.solutions
        ]
        raw = np.degrees(
            truth.rotation.distance_to(report.best().orientation)
        )
        print(f"{phase.name:<15} {str(uvw):<10} {raw:9.3f} deg {deviations[0]:11.2e} deg "
              f"{min(deviations):9.2e} deg {len(deviations):7d}")

Two different things are visible here, and separating them matters.

**The "raw angle" column is measurement, not error.** The recovered rotation is routinely tens
of degrees from the one the simulator used, while the disorientation is zero: the difference
is a crystal symmetry operation, and the two rotations describe the same crystal in the same
place. Comparing orientations without reducing by symmetry is the most common way to convince
yourself that a correct indexing routine is broken.

**But look at Ni $[112]$.** Its best-ranked solution is *not* zero even after symmetry
reduction — while the "best of all" column is. The true answer is in the report; it simply is
not the one ranked first, because nothing in the pattern distinguishes them. That is not a
symmetry-reduction subtlety. It is Friedel's law again, and it deserves its own look.

In [ ]:
table, _, report = round_trip(NICKEL, (1, 1, 2), [NICKEL])
true_matrix = np.asarray(table.basis).T

print(f"{len(report.solutions)} solutions returned, both indexing every spot exactly:\n")
for rank, solution in enumerate(report.solutions):
    relative = np.asarray(solution.orientation.as_matrix()) @ true_matrix.T
    angle = np.degrees(np.arccos(np.clip((np.trace(relative) - 1.0) / 2.0, -1.0, 1.0)))
    eigenvalues, eigenvectors = np.linalg.eig(relative)
    axis = np.real(eigenvectors[:, np.argmin(np.abs(eigenvalues - 1.0))])
    print(f"  rank {rank}: {solution.zone_axis_label}, "
          f"residual {solution.mean_residual_inv_angstrom:.1e} 1/A, "
          f"differs from the truth by {angle:6.2f} deg about "
          f"[{axis[0]:+.3f} {axis[1]:+.3f} {axis[2]:+.3f}] in the pattern frame")

The two solutions differ by **180 degrees about the beam** — the third axis of the pattern
frame. That rotation maps the two-dimensional spot set onto itself for *every* pattern,
because Friedel's law makes the set centrosymmetric: if $\mathbf{g}$ is there, so is
$-\mathbf{g}$.

Whether that leaves a real ambiguity depends on the zone. For Ni $[001]$, $[011]$ and $[111]$
a half turn about the zone axis *is* a cubic symmetry operation, so the two descriptions are
the same orientation and the disorientation is zero. For $[112]$ it is not — $\langle 112
\rangle$ is not a two-fold axis of $m\bar{3}m$ — so the two solutions are genuinely different
crystal orientations producing an identical pattern, and no amount of care with a single
pattern will choose between them.

This is exactly what `pytex.tem.ambiguity.analyze_ambiguity` enumerates, and what
`pytex.tem.indexing.orientation_from_indexed_pattern` reports as
`equivalent_orientations` rather than silently picking one. Tutorial 24 follows that thread.

## 8. Noise: where the method stops working

Real spot positions are picked by eye or by a centroid algorithm, and they are wrong by a
fraction of a percent at best. The two tolerances in the solver exist to absorb that, and the
question is how much they can absorb.

Below, Gaussian noise of increasing width is added to every spot position, and the pattern is
re-solved at the default tolerances.

In [ ]:
rng = np.random.default_rng(20260809)
noise_levels_mm = np.array([0.0, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0])
trials = 24

results = {}
for phase, uvw in ((NICKEL, (0, 1, 1)), (ZIRCONIUM, (1, 1, 0))):
    matched, correct, residual = [], [], []
    for noise in noise_levels_mm:
        fractions, hits, residuals = [], [], []
        for _ in range(trials):
            _, _, report = round_trip(phase, uvw, [phase], rng=rng, noise_mm=float(noise))
            if not report.solutions:
                fractions.append(0.0); hits.append(0.0); continue
            best = report.best()
            fractions.append(best.matched_fraction)
            hits.append(float(zone_equivalent(
                phase, uvw, np.asarray(best.zone_axis.indices, dtype=np.int64))))
            residuals.append(best.mean_residual_inv_angstrom)
        matched.append(np.mean(fractions))
        correct.append(np.mean(hits))
        residual.append(np.mean(residuals) if residuals else np.nan)
    results[phase.name] = (np.array(matched), np.array(correct), np.array(residual))

fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.0))
for name, (matched, correct, residual) in results.items():
    axes[0].plot(noise_levels_mm, 100 * correct, "o-", label=f"{name}: correct zone")
    axes[0].plot(noise_levels_mm, 100 * matched, "s--", alpha=0.6, label=f"{name}: spots indexed")
    axes[1].plot(noise_levels_mm, residual, "o-", label=name)
axes[0].set_xlabel("spot-position noise (mm, 1 sigma)")
axes[0].set_ylabel("percent")
axes[0].set_title("indexing quality against picking noise")
axes[0].legend(fontsize=8)
axes[1].set_xlabel("spot-position noise (mm, 1 sigma)")
axes[1].set_ylabel("mean residual (1/A)")
axes[1].set_title("residual grows with the noise it is measuring")
axes[1].legend(fontsize=8)
plt.show()

print("Innermost spot radius, for scale:")
for phase, uvw in ((NICKEL, (0, 1, 1)), (ZIRCONIUM, (1, 1, 0))):
    radii = np.linalg.norm(np.asarray(simulate(phase, uvw).detector_mm), axis=1)
    print(f"  {phase.name:<15} {radii.min():.1f} mm")

The residual is the useful output here. It is not a fitted parameter — it is the distance
between each measured spot and where the solution says it should be, so it *measures* the
picking error rather than hiding it. A solution whose residual is far below the noise you know
you have is a solution that has over-fitted; one whose residual matches the picking scatter is
behaving.

Note also which curve falls first. The fraction of spots indexed degrades gracefully, while
the zone-axis identification either survives or does not: the zone axis comes from the two
*seed* vectors, so once the noise is large enough to mis-seed, the rest of the answer is
irrelevant.

## 9. Phase discrimination, and the case where one pattern is not enough

The solver takes a list of candidate phases and ranks across all of them. Nickel and zirconium
are easy to separate — a cubic and a hexagonal metric share very few $d$-spacing ratios — so
this is the case that *should* work.

In [ ]:
print(f"{'true phase':<15} {'zone':<10} {'best':<28} {'conclusive':>11}")
for phase, uvw in ((NICKEL, (0, 0, 1)), (NICKEL, (1, 1, 1)),
                   (ZIRCONIUM, (0, 0, 1)), (ZIRCONIUM, (1, 1, 0))):
    _, _, report = round_trip(phase, uvw, [NICKEL, ZIRCONIUM])
    best = report.best()
    print(f"{phase.name:<15} {str(uvw):<10} "
          f"{best.phase_name + ' ' + best.zone_axis_label:<28} {str(report.is_conclusive):>11}")

Now the failure. Strip a pattern down to its two innermost spots — the situation when only the
strongest reflections are visible — and loosen the length tolerance from the 3 percent default
to the 5 percent a hand-picked micrograph really deserves.

In [ ]:
table = simulate(NICKEL, (1, 1, 1))
coordinates = np.asarray(table.detector_mm)
keep = np.argsort(np.linalg.norm(coordinates, axis=1))[:2]
sparse = MeasuredSAEDPattern(
    name="ni_111_two_spots",
    spots=tuple(MeasuredSpot(position=(float(x), float(y))) for x, y in coordinates[keep]),
    calibration=CALIBRATION,
)

for tolerance in (0.03, 0.05):
    report = solve_saed_pattern(
        sparse, [NICKEL, ZIRCONIUM], max_index=4, length_tolerance_relative=tolerance
    )
    distinct = sorted({(s.phase_name, s.zone_axis_label) for s in report.solutions})
    print(f"length tolerance {100 * tolerance:.0f}%: conclusive = {report.is_conclusive}, "
          f"distinct answers = {distinct}")

print()
print(solve_saed_pattern(
    sparse, [NICKEL, ZIRCONIUM], max_index=4, length_tolerance_relative=0.05
).describe())

At 3 percent the two spots still pick nickel out. At 5 percent — which is *not* a pessimistic
number for spots picked by eye — a zirconium zone explains them just as well, and the verdict
flips. Two spots are the minimum the type accepts and they are enough to *seed* a solution, but
they over-determine nothing, so nothing is left to break the tie.

This is the behaviour to want: `is_conclusive` is a claim about **discrimination**, not about
whether a number was produced. It is also a reminder that the tolerance is not a nuisance
parameter — set it to what your picking actually achieves, because setting it tighter buys a
confident answer that the data does not support.

The remedy in the microscope is the same as the remedy here: tilt to a second zone axis. Two
zones fix the orientation completely, and they also determine the diffraction rotation that a
single pattern leaves free — see tutorial 24, *TEM tilt navigation*, and
`pytex.tem.indexing.orientation_from_indexed_patterns`.

## 10. What to take away

- **Round-trip your inverse problems.** Simulate from a known answer, solve, and compare *up
  to the symmetry the measurement cannot break*. It is cheap, and it catches sign and
  convention errors that no amount of reading catches.
- **Compare orientations symmetry-aware, always.** Section 7's raw-angle column is what a
  naive comparison reports for a perfectly correct answer.
- **A recovered zone axis is a symmetry orbit, not a triple.** And its sense is undetermined
  by one pattern, by Friedel's law.
- **Residuals measure your picking, not the fit.** They are the honest quality signal.
- **`is_conclusive` is the number to read.** A solver always returns its best guess; only the
  verdict tells you whether the pattern discriminated.

### Further reading

- `docs/tex/algorithms/saed_ratio_angle_indexing.tex` — the indexing algorithm in full.
- `docs/tex/algorithms/reciprocal_space_and_kinematic_spots.tex` — the forward model.
- Tutorial 12, *SAED workflows* — the picking front end and the YAML pattern contract.
- Tutorial 24, *TEM tilt navigation* — from an indexed pattern to a crystal orientation, and
  why two patterns are needed.
- Tutorial 28, *CBED analysis* — what a convergent probe adds that a parallel beam cannot give.
- Williams and Carter, *Transmission Electron Microscopy*, 2nd ed., Part 2 — diffraction
  pattern indexing in practice.